### Importation des bibliothèques nécessaires

```python

In [47]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import numpy as np

In [48]:
# Récupération du contenu de la page d'acceuil du site de vente
url="https://www.europe-camions.com/tracteur-occasion/1-31/annonces-tracteur.html"
contenu_site = requests.get(url)
extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 
"""
On définit un dictionnnaire vide ayant comme clé les caractérique de la publication que nous cherchons à scraper.
"""
data = {"Réf. client :" :[], "Description": [], "Kilométrage": [], "Date de 1ere immatriculation": [], "Energie": [],
        "Marque":[], "Modèle":[], "Pays":[], "Prix":[], "Vendeur":[], "Localisation":[], "Photo":[],
        "Poids à vide":[], "PTC":[], "Essieux":[], "Puissance":[], "Etat":[]}

# Première page
"""_summary_
Après inspection de la page, nous remarquons que les blocs d'informations que nous cherchons à scraper sont tous regroupés dans des div de classe "google-tag". 
Nous allons donc parcourir tous ces blocs et extraire les informations qui nous intéressent.
"""
info_vehicule = extract_contenu.find_all('div', class_='google-tag') 
for info_pub in info_vehicule:
    
    prix = info_pub.find("span", class_="meta-price")
    if prix:
        data["Prix"].append(prix.get_text(strip=True))
    else:
        data["Prix"].append(np.nan)
    
    vendeur = info_pub.find("span", class_ ="title-vendeur")
    if vendeur:
        data["Vendeur"].append(vendeur.get_text(strip=True))
    
    
    geo = info_pub.find("span", class_="d-none")
    if geo:
        data["Localisation"].append(geo.get_text(strip=True).replace("-", ""))
    
    
    titre = info_pub.select_one("h2")
    if titre:
        data["Description"].append(titre.get_text(strip=True))
    
    img = info_pub.select_one("img")
    if img and img.has_attr("src"):
        data["Photo"].append(img.get("src"))

    """
    Lorsqu'on déroule les éléments de page, on voit que les info sont contenues dans des balises td et que toutes les caractéristiques sont composées par deux td succesives
    td 1 : caractéristique
    td 2 : valeur
    Nous ce que intéresse c'est le td 2, pour cela on parcours toutes td et on met une condition sur le td à l'instant t.
    Si la valeur du td est dans la liste des clées de notre dictionnaires alors forcément le td suivant qui est la valeurs fait partie des caractéristiques à scarper.
    """
    description = info_pub.find_all("td")
    for index, value in enumerate(description):
        # si la balise td courante contient une des caractéristiques que nous cherchons à scraper alors on ajoute la valeur du td suivant à notre dictionnaire
        if value.get_text(strip=True) in data.keys():
            value_suivante = description[index + 1]
            if value_suivante:
                data[value.get_text(strip=True)].append(value_suivante.get_text(strip=True))
            continue

In [49]:
pages_vues = set("https://www.europe-camions.com/tracteur-occasion/1-31/annonces-tracteur.html")
while url:
    if url in pages_vues:  # si on a déjà visité cette page, on arrête
        print(f"Page déjà vue : {url}, fin du scraping")
        break
    pages_vues.add(url)  # ajouter la page actuelle à l'ensemble des pages vues
    contenu_site = requests.get(url)
    
    extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 

    info_vehicule = extract_contenu.find_all('div', class_='google-tag') 
    for info_pub in info_vehicule:
        
        prix = info_pub.find("span", class_="meta-price")
        if prix:
            data["Prix"].append(prix.get_text(strip=True))
        
        vendeur = info_pub.find("span", class_ ="title-vendeur")
        if vendeur:
            data["Vendeur"].append(vendeur.get_text(strip=True))
        
        
        geo = info_pub.find("span", class_="d-none")
        if geo:
            data["Localisation"].append(geo.get_text(strip=True).replace("-", ""))
        
        
        titre = info_pub.select_one("h2")
        if titre:
            data["Description"].append(titre.get_text(strip=True))
        
        img = info_pub.select_one("img")
        if img and img.has_attr("src"):
            data["Photo"].append(img.get("src"))

        
        description = info_pub.find_all("td")
        for index, value in enumerate(description):
            # si la balise td courante contient une des caractéristiques que nous cherchons à scraper alors on ajoute la valeur du td suivant à notre dictionnaire
            if value.get_text(strip=True) in data.keys():
                value_suivante = description[index + 1]
                if value_suivante:
                    data[value.get_text(strip=True)].append(value_suivante.get_text(strip=True))
                continue
            

   
    #Pause pour ne pas surcharger le serveur
    time.sleep(0.5)
    # Pagination : chercher le bouton "next"
    next_button = extract_contenu.select_one("div.btn-next-pagination.next.btn.text-center.no-decoration")
    if next_button and next_button.has_attr("data-ihref"):
        next_page = next_button["data-ihref"]
        print(f"Page suivante trouvée : {next_page}")
        url = "https://www.europe-camions.com" + next_page
    else:
        url = None

Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=2
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=3
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=4
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=5
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=6
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=7
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=8
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=9
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=10
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=11
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=12
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=13
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=14
Page suivante trouvé

In [115]:
df = pd.DataFrame(data.values(), index=data.keys()).T
df["Puissance"] = df["Puissance"].replace(r"[^\d]", "", regex=True)
df.head()

,Réf. client :,Description,Kilométrage,Date de 1ere immatriculation,Energie,Marque,Modèle,Pays,Prix,Vendeur,Localisation,Photo,Poids à vide,PTC,Essieux,Puissance,Etat
0,00013161,Tracteur Mercedes Arocs 2046,220 000 km,31/01/2018,Gazoil,Mercedes,2046,FRANCE,Prix sur demande,BERROYER,Vendée,https://photo.static-viamobilis.com/31/1/81031...,7.51 Tonnes,19 Tonnes,4x2,460,Occasion
1,00013489,Tracteur DAF XF 530,538 000 km,18/12/2019,Gazoil,DAF,530,FRANCE,45 800 EURHT,BERROYER,Vendée,https://photo.static-viamobilis.com/31/1/89923...,8.27 Tonnes,19 Tonnes,4x2,530,Occasion
2,VO25-1966,Tracteur Volvo F6 460 FH,511 000 km,04/04/2019,Gazoil,Volvo,1843,FRANCE,Prix sur demande,SYLTRAILER,PyrénéesOrientales,https://photo.static-viamobilis.com/31/1/10861...,7.65 Tonnes,19 Tonnes,4x2,460,Occasion
3,VO26-2072,Tracteur Iveco S-WAY 490,416 350 km,21/10/2022,Gazoil,Iveco,2046,FRANCE,Prix sur demande,SYLTRAILER,PyrénéesOrientales,https://photo.static-viamobilis.com/31/1/11115...,6.87 Tonnes,18 Tonnes,4x2,490,Occasion
4,VO25-1984,Tracteur produits dangereux / adr Renault neuf...,74 km,12/06/2024,Gazoil,Renault,530,FRANCE,Prix sur demande,SYLTRAILER,PyrénéesOrientales,https://photo.static-viamobilis.com/31/1/10906...,8.58 Tonnes,19 Tonnes,4x2,480,Neuf


In [ ]:
## Après inspection de la colonne prix, on remarque que les prix sont exprimés de différentes façons :
# - Certains prix sont exprimés en euros par mois pour les véhicules en location
# - D'autres sont indiqués comme "Prix sur demande" ou "Tarif de location sur demande"
# - Les autres sont des prix d'achat exprimés en euros
# Nous allons donc créer quatre DataFrame distincts pour chaque type de prix afin de les traiter

## Véhicules en location avec un prix exprimé en euros par mois
LocationParMois = df[df["Prix"].str.contains("mois en location", na=False)]
LocationParMois["Prix"] = LocationParMois["Prix"].replace(r"[^\d]", "", regex=True)

## Véhicules avec un prix exprimé comme "Prix sur demande"
PrixSurDemande = df[df["Prix"].str.contains("Prix sur demande", na=False)]

## Véhicules avec un prix exprimé comme "Tarif de location sur demande"
PrixLocationSurDemande = df[df["Prix"].str.contains("Tarif de location sur demande", na=False)]

## Véhicules avec un prix d'achat exprimé en euros
Achat = df[~df["Prix"].str.contains("mois en location|Prix sur demande|Tarif de location sur demande", na=False)]
Achat["Prix"] = Achat["Prix"].replace(r"[^\d]", "", regex=True)

/tmp/ipykernel_782744/2821931391.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  LocationParMois["Prix"] = LocationParMois["Prix"].replace(r"[^\d]", "", regex=True)
/tmp/ipykernel_782744/2821931391.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Achat["Prix"] = Achat["Prix"].replace(r"[^\d]", "", regex=True)


In [ ]:
## Base de données des véhicules sous location par mois
LocationParMois.to_csv("LocationParMois.csv", index=False)

## Base de données des véhicules avec prix sur demande
PrixSurDemande.to_csv("PrixSurDemande.csv", index=False)

## Base de données des véhicules avec tarif de location sur demande
PrixLocationSurDemande.to_csv("PrixLocationSurDemande.csv", index=False)

## Base de données des véhicules à vendre
Achat.to_csv("Achat.csv", index=False)